## Finding all adverbials from a database

The aim of this notebook is to find all adverbials from the Estonian Reference corpus. This is needed to annotate them with semantic class using both rule based methods and LLMs. The code extracts data from Katrin Tsepelina's database [v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db](https://github.com/estnltk/syntax_experiments/tree/verb_templates/workflows/001_verb_transactions/v33) with data extracted from the Estonian Reference corpus.

This code creates a 2 new tables in the database
1. **spatial_obl** with nominal adverbials aka obliques in spatial cases (form + lemma + feats), their head verb (verb+compund) and sentences the obliques came from.
2. **advmod** with adverbs (form + lemma), their head verbs (verb+compound) and sentences the adverbs came from

The sentences are taken from another database and added to this one based on their sentence id.

In [1]:
#imports
import sqlite3

In [2]:
# database file path
filename = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"

# connecting with database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

### Create new table spatial_obl

In [3]:
cursor.execute("DROP TABLE spatial_obl")

In [4]:
#searches for obliques in spatial cases + head verb + sentence id
query = (f"CREATE TABLE spatial_obl AS " 
         f"SELECT transaction_row.id, transaction_row.head_id, transaction_row.form, lemma, transaction_row.feats, pos, verb, verb_compound, sentence_id " 
         f"FROM `transaction_row` JOIN `transaction_head` ON transaction_head.id = transaction_row.head_id WHERE transaction_row.deprel = 'obl' "
        f"AND (transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ?)")
cursor.execute(query, ('%adit,%', '%ill,%', '%in,%', '%el,%', '%all,%', '%ad,%', '%abl,%'))

### Create new table advmod

In [3]:
cursor.execute("DROP TABLE advmod")

In [4]:
#searches for obliques in spatial cases + head verb + sentence id
query = (f"CREATE TABLE advmod AS " 
         f"SELECT transaction_row.id, transaction_row.head_id, transaction_row.form, transaction_row.lemma, pos, verb, verb_compound, sentence_id " 
         f"FROM `transaction_row` JOIN `transaction_head` ON transaction_head.id = transaction_row.head_id WHERE transaction_row.deprel = 'advmod' ")
cursor.execute(query)

### Add sentences to tables

In [5]:
def column_exists(cursor, table, column):
    cursor.execute(f"PRAGMA table_info({table})")
    return any(row[1] == column for row in cursor.fetchall())

In [6]:
# Connect to both databases
filename1 = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_sentences_20250220-130121.db"
filename2 = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"

conn_source = sqlite3.connect(filename1)  # Source database
conn_target = sqlite3.connect(filename2)  # Target database

cursor_source = conn_source.cursor()
cursor_target = conn_target.cursor()

In [7]:
def sentences_to_table(input_table, cursor1, cursor2, conn1, conn2):
    # Step 1: Retrieve sentences from database1
    cursor1.execute("SELECT id, text FROM sentences")
    sentences = cursor1.fetchall()  # List of (sentence_id, sentence)

    # Step 2: add new column to database2 table
    if not column_exists(cursor2, input_table, "sentence"):
        cursor2.execute("ALTER TABLE " + input_table +  " ADD COLUMN sentence TEXT")

    # Step 3: Create a temporary table
    cursor2.execute("CREATE TEMP TABLE temp_sentence (id INT PRIMARY KEY, sentence TEXT)")

    # Step 4: Insert all values into the temp table
    cursor2.executemany("INSERT INTO temp_sentence (id, sentence) VALUES (?, ?)", sentences)

    # Step 3: Perform a fast join-based update
    query = f"""
        UPDATE {input_table}
        SET sentence = (
            SELECT sentence
            FROM temp_sentence
            WHERE temp_sentence.id = {input_table}.sentence_id
            LIMIT 1
         )
        WHERE EXISTS (
            SELECT 1
            FROM temp_sentence
            WHERE temp_sentence.id = {input_table}.sentence_id
        )
    """

    cursor2.execute(query)

    conn2.commit()
    conn1.close()
    conn2.close()

In [8]:
sentences_to_table('advmod', cursor_source, cursor_target, conn_source, conn_target)

In [8]:
sentences_to_table('spatial_obl', cursor_source, cursor_target, conn_source, conn_target)